# Radionica 3: Hoće li korisnik otići?
**Klasifikacija i evaluacija modela na primjeru telekom operatera**

Mi smo telekom operater. Svaki mjesec dio korisnika raskine ugovor i pređe konkurenciji (*churn*).
Želimo naći korisnike koji će vjerovatno otići i poslati im **email s 25 % popusta** prije nego što odu.

**Podaci:** 7.043 korisnika (Telco Customer Churn, IBM-ov primjer podataka). Za svakog znamo koje usluge koristi, kakav ugovor ima, koliko plaća i da li je otišao.

**Kako radimo:** ćelije s `# TODO` popunjavate vi. Gdje piše `___`, tu nešto nedostaje.
Dijelovi označeni s ⭐ su bonus, ako stignete.

## 0. Priprema

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

print("Sve učitano!")

## 1. Priprema podataka

Pravi podaci nikad nisu uredni. Prvo ih "počešljamo".

In [ ]:
df = pd.read_csv("churn.csv")
df.head()

In [ ]:
# Nazivi kolona i vrijednosti: mala slova, bez razmaka
df.columns = df.columns.str.lower().str.replace(" ", "_")

tekstualne = df.select_dtypes(exclude="number").columns
for c in tekstualne:
    df[c] = df[c].str.lower().str.replace(" ", "_")

df.head().T

**Zamka u podacima:** kolona `totalcharges` (ukupno plaćeno) trebala bi biti broj, ali pandas je čita kao tekst. Zašto?

In [ ]:
tc = pd.to_numeric(df.totalcharges, errors="coerce")
df[tc.isnull()][["customerid", "tenure", "totalcharges"]]

Ovi korisnici imaju `tenure = 0`: tek su došli i još ništa nisu platili, pa umjesto broja stoji prazan razmak (koji je naše čišćenje pretvorilo u `_`).
Pretvaramo kolonu u broj, a prazna mjesta popunjavamo nulom.

In [ ]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors="coerce")
df.totalcharges = df.totalcharges.fillna(0)

Target je kolona `churn` (`yes`/`no`). Model treba brojeve, pa pravimo 1 = otišao, 0 = ostao.

In [ ]:
# TODO: pretvorite churn u 0 i 1
# Pomoć: (df.churn == "yes") daje True/False, a .astype(int) to pretvara u 1/0
df.churn = (df.churn == ___).astype(int)
df.churn.head()

## 2. Train, validacija i test

Tri skupa, kao na prošloj radionici:
- **train (60 %)**: model uči
- **validacija (20 %)**: probni kolokvij, biramo model i postavke
- **test (20 %)**: ispit, gledamo ga tek na kraju

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

len(df_train), len(df_val), len(df_test)

**Pitanje:** zašto `test_size=0.25` u drugom redu, a ne 0.2? (Pomoć: 25 % od 80 % je…)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

# Odgovor izbacujemo iz ulaza, inače bi model "varao"
del df_train["churn"]
del df_val["churn"]
del df_test["churn"]

## 3. Ko odlazi? (EDA)

Koristimo `df_full_train` (train + validacija), a test ne diramo.

In [ ]:
# TODO: koliki procenat korisnika ode?
# Pomoć: prosjek kolone sa 0 i 1 je udio jedinica
global_churn = df_full_train.churn.___()
print(f"Ode {global_churn * 100:.1f} % korisnika")

In [ ]:
numerical = ["tenure", "monthlycharges", "totalcharges"]

categorical = [
    "gender", "seniorcitizen", "partner", "dependents",
    "phoneservice", "multiplelines", "internetservice",
    "onlinesecurity", "onlinebackup", "deviceprotection", "techsupport",
    "streamingtv", "streamingmovies", "contract", "paperlessbilling",
    "paymentmethod",
]

### Churn rate i rizik po grupama

Za svaku grupu gledamo koliko njih ode i poredimo s prosjekom:
- **diff** = grupa − prosjek (pozitivno: odlaze češće)
- **risk** = grupa / prosjek (veće od 1: rizičnija grupa)

In [ ]:
# Primjer: spol
df_full_train.groupby("gender").churn.mean()

In [ ]:
# TODO: isto za vrstu ugovora (contract)
df_full_train.groupby(___).churn.mean()

In [ ]:
from IPython.display import display

for c in ["contract", "partner", "internetservice", "techsupport", "paymentmethod", "gender"]:
    grupa = df_full_train.groupby(c).churn.agg(["mean", "count"])
    grupa["diff"] = grupa["mean"] - global_churn
    grupa["risk"] = grupa["mean"] / global_churn
    print(c)
    display(grupa.round(3))

**Pitanje:** koja grupa je najrizičnija? A koja kolona skoro ništa ne govori o odlasku?

### Mutual information: jedan broj za cijelu kolonu

Koliko nam poznavanje kolone govori o tome da li će korisnik otići. Veće je bolje.

In [ ]:
def mi_churn(kolona):
    return mutual_info_score(kolona, df_full_train.churn)

mi = df_full_train[categorical].apply(mi_churn)
mi.sort_values(ascending=False)

### Brojčane kolone: korelacija

In [ ]:
df_full_train[numerical].corrwith(df_full_train.churn)

In [ ]:
# TODO: koliko odlaze novi korisnici (tenure <= 2 mjeseca), a koliko stari (tenure > 12)?
print("Novi:", df_full_train[df_full_train.tenure <= ___].churn.mean().round(3))
print("Stari:", df_full_train[df_full_train.tenure > ___].churn.mean().round(3))

**Pitanje:** korelacija `tenure` je negativna. Šta to znači riječima?

## 4. One-hot encoding

Model razumije samo brojeve. Kategoriju (npr. ugovor) pretvaramo u kolone s 0 i 1, po jednu za svaku vrijednost.
Danas koristimo `DictVectorizer` iz scikit-learna.

### Korak po korak: prvo dvije kolone i 10 korisnika

`DictVectorizer` prima listu rječnika: svaki korisnik je jedan rječnik `{kolona: vrijednost}`.

In [ ]:
dicts = df_train[["gender", "contract"]].iloc[:10].to_dict(orient="records")
dicts

In [ ]:
dv = DictVectorizer()
dv.fit(dicts)                 # fit: dv pogleda podatke i nauči koje kolone treba napraviti
dv.get_feature_names_out()

In [ ]:
dv.transform(dicts)           # transform: pretvori korisnike u brojeve

Dobile smo neku "sparse matrix". To je štedljiv zapis: pamte se samo jedinice, a nule se ne pamte.
Korisno kad ima mnogo nula, ali mi želimo vidjeti cijelu tabelu. Zato dodajemo `sparse=False`.

In [ ]:
# TODO: napravite DictVectorizer koji vraća običnu tabelu
dv = DictVectorizer(sparse=___)
dv.fit(dicts)
dv.transform(dicts)

In [ ]:
# Ista tabela, s nazivima kolona, da lakše čitamo
pd.DataFrame(dv.transform(dicts), columns=dv.get_feature_names_out()).astype(int)

**Pitanje:** prvi korisnik ima 1 u kolonama `contract=two_year` i `gender=female`. Šta to znači?

**Pitanje 2:** gdje je kolona `contract=one_year`? (Pomoć: pogledajte koje ugovore imaju ovih 10 korisnika.
`DictVectorizer` pravi kolone samo za vrijednosti koje vidi u `fit`. Zato ga učimo na cijelom train skupu.)

### Dodajmo i jednu brojčanu kolonu

In [ ]:
# TODO: dodajte kolonu tenure i pogledajte šta se desi s njom
dicts = df_train[["gender", "contract", ___]].iloc[:10].to_dict(orient="records")
dv = DictVectorizer(sparse=False)
dv.fit(dicts)
pd.DataFrame(dv.transform(dicts), columns=dv.get_feature_names_out())

**Pitanje:** zašto `tenure` ostaje jedna kolona, a `contract` postaje više njih?

### Sve kolone, svi korisnici

Sad isto za sve kolone. Na train skupu `fit_transform` (nauči kolone i pretvori), a na validaciji samo `transform`.

In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient="records")
X_val = dv.transform(val_dict)   # samo transform: kolone naučene na train skupu

X_train.shape

## 5. Logistička regresija

Isto kao linearna regresija (težine × kolone + slobodni član), samo rezultat provučemo kroz **sigmoid** i dobijemo broj od 0 do 1: vjerovatnoću odlaska.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-7, 7, 51)
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", linestyle="--")
plt.xlabel("rezultat linearnog dijela (z)")
plt.ylabel("vjerovatnoća")
plt.show()

**Pitanje:** šta sigmoid vrati za z = 0? A za jako veliki pozitivni ili negativni z?

In [ ]:
# TODO: istrenirajte logističku regresiju
model = LogisticRegression()
model.___(X_train, y_train)

In [ ]:
# Vjerovatnoća odlaska za svakog korisnika iz validacije
y_pred = model.predict_proba(X_val)[:, 1]
y_pred[:10].round(2)

In [ ]:
# TODO: kome šaljemo popust? Svima s vjerovatnoćom >= 0.5
churn_decision = (y_pred >= ___)

print(f"Accuracy: {(churn_decision == y_val).mean() * 100:.1f} %")

## 6. Šta je model naučio?

Svaka kolona ima svoju težinu. Pozitivna težina gura vjerovatnoću gore (veći rizik), negativna dolje.

In [ ]:
tezine = pd.Series(model.coef_[0], index=dv.get_feature_names_out())
tezine.sort_values().round(3)

**Pitanje:** koja kolona najviše povećava rizik odlaska, a koja ga najviše smanjuje? Slaže li se to s tabelom rizika iz koraka 3?

### Mali model koji možemo izračunati ručno

In [ ]:
small = ["contract", "tenure", "monthlycharges"]

dv_small = DictVectorizer(sparse=False)
X_train_small = dv_small.fit_transform(df_train[small].to_dict(orient="records"))

model_small = LogisticRegression()
model_small.fit(X_train_small, y_train)

w0 = model_small.intercept_[0]
w = {k: float(v) for k, v in zip(dv_small.get_feature_names_out(), model_small.coef_[0].round(3))}
print("w0 =", round(w0, 3))
w

In [ ]:
# TODO: korisnik s ugovorom na dvije godine, 24 mjeseca kod nas, plaća 30 mjesečno.
# Izračunajte z = w0 + težina ugovora + 30 * težina računa + 24 * težina staža, pa sigmoid.
z = w0 + w["contract=two_year"] + 30 * w[___] + 24 * w[___]
print(f"Vjerovatnoća odlaska: {sigmoid(z):.3f}")

In [ ]:
# ⭐ Probajte isto za korisnika s mjesečnim ugovorom, 1 mjesec kod nas, plaća 80 mjesečno


## 7. Je li 80 % dobro? (evaluacija)

Accuracy je oko 80 %. Zvuči dobro. Ali uporedimo s modelom koji ništa ne radi.

In [ ]:
# TODO: model koji kaže da NIKO ne odlazi. Koliko je tačan?
# Pomoć: np.zeros(n) daje niz od n nula
niko_ne_odlazi = np.zeros(___)
print(f"Accuracy modela 'niko ne odlazi': {(niko_ne_odlazi == y_val).mean() * 100:.1f} %")

Naš model je bolji za samo nekoliko procenata. Accuracy ne kaže **gdje** model griješi.

### Matrica konfuzije

In [ ]:
stvarno_odlazi = (y_val == 1)
stvarno_ostaje = (y_val == 0)
kaze_odlazi = (y_pred >= 0.5)
kaze_ostaje = (y_pred < 0.5)

tp = (kaze_odlazi & stvarno_odlazi).sum()   # popust dobio onaj ko bi otišao
fp = (kaze_odlazi & stvarno_ostaje).sum()   # popust dobio onaj ko bi ionako ostao
fn = (kaze_ostaje & stvarno_odlazi).sum()   # otišao, a nismo mu ništa ponudili
tn = (kaze_ostaje & stvarno_ostaje).sum()

pd.DataFrame([[tn, fp], [fn, tp]],
             index=["stvarno ostaje", "stvarno odlazi"],
             columns=["model: ostaje", "model: odlazi"])

### Precision i recall

In [ ]:
# TODO: izračunajte precision i recall
# Precision = TP / (TP + FP),  Recall = TP / (TP + FN)
precision = tp / (___)
recall = tp / (___)

print(f"Precision: {precision:.2f}  -> od svih kojima šaljemo popust, toliko bi zaista otišlo")
print(f"Recall:    {recall:.2f}  -> od svih koji odlaze, toliko smo ih uhvatile")

**Diskusija:** šta je skuplje za operatera, popust poslan nekome ko bi ionako ostao (FP) ili korisnik koji ode bez ponude (FN)?

### Prag: kome slati popust?

0.5 nije zakon. Probajmo druge pragove.

In [ ]:
for prag in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    odluka = (y_pred >= prag)
    tp = (odluka & stvarno_odlazi).sum()
    fp = (odluka & stvarno_ostaje).sum()
    fn = (~odluka & stvarno_odlazi).sum()
    acc = (odluka == y_val).mean()
    print(f"prag {prag:.1f}: emailova {tp + fp:4d}  uhvaćeno {tp:3d}  propušteno {fn:3d}  "
          f"bespotrebnih popusta {fp:3d}  accuracy {acc:.3f}")

**Pitanje:** koji prag biste izabrale? Šta se dešava s brojem emailova kad spuštamo prag?

## 8. Završni ispit: test skup

Model treniramo na train + validaciji i jednom provjerimo na testu.

In [ ]:
dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(df_full_train[categorical + numerical].to_dict(orient="records"))
y_full_train = df_full_train.churn.values

model = LogisticRegression()
model.fit(X_full_train, y_full_train)

dicts_test = df_test[categorical + numerical].to_dict(orient="records")
X_test = dv.transform(dicts_test)
y_pred_test = model.predict_proba(X_test)[:, 1]

print(f"Accuracy na testu: {((y_pred_test >= 0.5) == y_test).mean() * 100:.1f} %")

In [ ]:
# Jedan korisnik iz testa: da li mu šaljemo popust?
korisnik = dicts_test[-1]
print(korisnik)
p = model.predict_proba(dv.transform([korisnik]))[0, 1]
print(f"Vjerovatnoća odlaska: {p:.2f}  ->  {'šaljemo popust' if p >= 0.5 else 'ne šaljemo'}")
print("Stvarno:", "otišao" if y_test[-1] == 1 else "ostao")

---
# ⭐ Bonus 

## Bonus A: koliko novca?

Recimo da popust od 25 % košta operatera **10 KM**, a korisnik koji ode znači gubitak od **100 KM**.
Pretpostavimo da svako ko dobije popust ostane.

In [ ]:
for prag in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    odluka = (y_pred >= prag)
    tp = (odluka & stvarno_odlazi).sum()
    fp = (odluka & stvarno_ostaje).sum()
    fn = (~odluka & stvarno_odlazi).sum()
    trosak = (tp + fp) * 10 + fn * 100
    print(f"prag {prag:.1f}: trošak {trosak:6d} KM")

**Pitanje:** koji prag je najjeftiniji? Je li to onaj s najvećim accuracy?

## Bonus B: random forest

Isti podaci, drugi model. Je li bolji?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict_proba(X_val)[:, 1]
print(f"Accuracy random forest: {((y_pred_rf >= 0.5) == y_val).mean() * 100:.1f} %")

## ⭐ Bonus C: cross-validation

Jedna podjela na train i validaciju može biti sreća ili nesreća.
Zato df_full_train podijelimo na 5 dijelova. Svaki dio jednom bude validacija,
a ostala četiri su train. Dobijemo 5 rezultata i gledamo prosjek.

In [ ]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=___, shuffle=True, random_state=1)
rezultati = []

for train_idx, val_idx in kfold.split(df_full_train):
    d_train = df_full_train.iloc[train_idx]
    d_val = df_full_train.iloc[val_idx]

    dv = DictVectorizer(sparse=False)
    X_tr = dv.___(d_train[categorical + numerical].to_dict(orient="records"))
    X_va = dv.___(d_val[categorical + numerical].to_dict(orient="records"))

    model = LogisticRegression(max_iter=1000)
    model.fit(X_tr, d_train.churn.values)

    y_p = model.predict_proba(X_va)[:, 1]
    rezultati.append(((y_p >= 0.5) == d_val.churn.values).mean())

print(np.round(rezultati, 3))
print(f"Prosjek: {np.mean(rezultati):.3f}  ±  {np.std(rezultati):.3f}")

### ⭐ Bonus C2: koji C je najbolji? (hyperparameter tuning)

`C` je hiperparametar logističke regresije: mi ga biramo, model ga ne uči.
Mali C znači oprezniji model s malim težinama, a veliki C daje modelu više slobode.

Za svaku vrijednost C uradimo cross-validation kroz 5 dijelova i uporedimo prosjeke.
Test i dalje ne diramo.

In [ ]:
for C in [0.01, 0.1, 1, 10]:
    rezultati = []

    for train_idx, val_idx in kfold.split(df_full_train):
        d_train = df_full_train.iloc[train_idx]
        d_val = df_full_train.iloc[val_idx]

        dv = DictVectorizer(sparse=False)
        X_tr = dv.fit_transform(d_train[categorical + numerical].to_dict(orient="records"))
        X_va = dv.transform(d_val[categorical + numerical].to_dict(orient="records"))

        # TODO: postavite C
        model = LogisticRegression(C=___, max_iter=1000)
        model.fit(X_tr, d_train.churn.values)

        y_p = model.predict_proba(X_va)[:, 1]
        rezultati.append(((y_p >= 0.5) == d_val.churn.values).mean())

    print(f"C = {C:5}:  {np.mean(rezultati):.3f} ± {np.std(rezultati):.3f}")

**Pitanje:** koji C je najbolji? Je li razlika velika u poređenju s rasponom (±)?

---
## 📝 Zadaća

1. Pogledajte bonus A. Ako popust košta **30 KM**, pa **50 KM** umjesto 10 KM, koji prag je sad najjeftiniji? Objasnite u dvije rečenice zašto se mijenja.
2. Izbacite iz modela kolone s najmanjim mutual information (npr. `gender`, `phoneservice`, `multiplelines`). Mijenja li se accuracy na validaciji?